[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc3_stats/corrections/seance3_correction.ipynb)

# Séance 3.3 — Relier deux variables — y a-t-il un lien ?

**Correction** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- réutiliser `pd.cut` (vu en séance 2.2) pour fabriquer une variable qualitative
- tester le lien entre deux variables qualitatives avec un khi-deux
- lire un tableau d'effectifs attendus pour dire *où* est la dépendance
- mesurer un lien entre deux variables quantitatives (Pearson, Spearman)
- reconnaître les trois pièges de la corrélation : extrêmes, non-linéarité, variable de confusion

## Correction

Solutions commentées. Comparez avec ce que vous aviez écrit : plusieurs formulations peuvent être correctes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc3_stats/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
cmd = pd.read_csv(BASE + "commandes.csv")

# pd.cut decoupe une colonne continue en categories ; 1e9 = "et au-dela"
cmd["taille"] = pd.cut(cmd["ca"], [0, 200, 500, 1e9],
                       labels=["petite", "moyenne", "grande"])
print(cmd["taille"].value_counts())

---

# Partie 1 — L'échauffement

Le code est déjà écrit : il ne reste que les `____` à remplir. Allez vite, l'essentiel
de la séance est dans la partie 2.

### Exercice 1 — Le tableau croisé

> **Votre mission :**
> - Croiser `pays` (en lignes) et `taille` (en colonnes) pour les quatre pays les plus présents → `tab`.
> - Combien de grosses commandes irlandaises ? → `irl_grande`

In [ ]:
top4 = cmd["pays"].value_counts().head(4).index
sub = cmd.query("pays in @top4")

# index = lignes, colonnes = ce qu'on croise avec
tab = pd.crosstab(sub["pays"], sub["taille"])
irl_grande = tab.loc["Irlande", "grande"]   ## .loc[ligne, colonne]
print(tab)

In [ ]:
verifier("1 - grosses commandes irlandaises", irl_grande == 157,
         "pd.crosstab(lignes, colonnes) puis .loc['Irlande', 'grande']")

### Exercice 2 — Le khi-deux

> **Votre mission :**
> - Tester le lien entre pays et taille de commande sur `tab`.
> - Récupérer la p-value → `p_khi2`, et conclure au seuil de 5 % → `dependant` (`True`/`False`).
> - Rappel : les quatre noms à gauche du `=` se lisent **dans l'ordre**.

In [ ]:
# chi2_contingency renvoie quatre choses : la statistique, la p-value,
# les degres de liberte et le tableau des effectifs attendus
khi2, p_khi2, ddl, attendus = stats.chi2_contingency(tab)

dependant = p_khi2 < 0.05   ## le seuil usuel, comme pour le test t
print("p =", p_khi2, "| dependance :", dependant)

In [ ]:
verifier("2 - dependance pays / taille", bool(dependant) and p_khi2 < 1e-20,
         "passez le tableau croise a stats.chi2_contingency")

### Exercice 3 — Où est la dépendance ?

> **Votre mission :**
> - Construire le tableau des écarts (observé − attendu) → `ecarts`.
> - Quel est l'écart des grosses commandes britanniques ? → `ecart_uk` (arrondi à 1 décimale)

In [ ]:
att = pd.DataFrame(attendus, index=tab.index, columns=tab.columns)
ecarts = tab - att   ## observe moins attendu, case par case

# Negatif : il y a MOINS de grosses commandes britanniques que ce qu'on
# attendrait si le pays n'y etait pour rien
ecart_uk = round(ecarts.loc["Royaume-Uni", "grande"], 1)
print(ecarts.round(1))

In [ ]:
verifier("3 - ecart des grosses commandes britanniques", ecart_uk == -82.7,
         "observe moins attendu, donc tab - att")

### Exercice 4 — Les corrélations d'un coup

> **Votre mission :**
> - Afficher la matrice de corrélation de `ca`, `nart` et `qte`.
> - En extraire la corrélation entre `ca` et `qte` → `r_ca_qte` (arrondie à 3 décimales).

In [ ]:
print(cmd[["ca", "nart", "qte"]].corr().round(3))   ## la matrice

r_ca_qte = round(cmd["ca"].corr(cmd["qte"]), 3)   ## 0,848 : tres lie
print(r_ca_qte)

In [ ]:
verifier("4 - correlation ca / qte", r_ca_qte == 0.848,
         "df['ca'].corr(df['qte'])")

### Exercice 5 — Pearson contre Spearman

> **Votre mission :**
> - Calculer les deux corrélations entre `ca` et `nart` → `r_pearson` et `r_spear` (arrondies à 3 décimales).
> - Laquelle est la plus élevée, et pourquoi ?

In [ ]:
r_pearson = round(cmd["ca"].corr(cmd["nart"]), 3)   ## methode par defaut

# Spearman raisonne sur les RANGS : il ne demande pas que la relation
# soit droite, seulement qu'elle monte
r_spear = round(cmd["ca"].corr(cmd["nart"], method="spearman"), 3)

print("Pearson", r_pearson, "| Spearman", r_spear)

In [ ]:
verifier("5a - Pearson", r_pearson == 0.382, "c'est la methode par defaut")
verifier("5b - Spearman", r_spear == 0.671, "method='spearman'")

### Exercice 6 — Le poids de 1 % des lignes

> **Votre mission :**
> - Retirer les 1 % de commandes les plus grosses → `sans`.
> - Recalculer la corrélation entre `ca` et `nart` sur ce sous-ensemble → `r_sans` (arrondie à 3 décimales).

In [ ]:
# quantile(0.99) : le montant au-dessus duquel se trouvent les 1 % du haut
seuil = cmd["ca"].quantile(0.99)
sans = cmd.query("ca < @seuil")   ## 20 commandes en moins seulement

r_sans = round(sans["ca"].corr(sans["nart"]), 3)   ## 0,382 -> 0,489
print(len(cmd) - len(sans), "lignes retirees | correlation :", r_sans)

# 0,382 -> 0,489 en retirant 20 lignes sur 1 955.

In [ ]:
verifier("6 - correlation sans les extremes", r_sans == 0.489,
         "les 1 % du haut commencent au quantile 0.99")

### Exercice 7 — Le même chiffre, deux pays

> **Votre mission :**
> - Calculer la corrélation `ca` / `nart` **en Belgique** → `r_be`, puis **en Irlande** → `r_irl`.
> - Arrondir à 3 décimales. Comparez au 0,382 global.

In [ ]:
be = cmd.query("pays == 'Belgique'")
irl = cmd.query("pays == 'Irlande'")   ## deux clients seulement

r_be = round(be["ca"].corr(be["nart"]), 3)     ## 0,903
r_irl = round(irl["ca"].corr(irl["nart"]), 3)  ## 0,234
print("Belgique", r_be, "| Irlande", r_irl)

# 0,903 contre 0,234, pour un 0,382 global qui n'est le chiffre de
# personne. Un coefficient calcule sur un melange de populations
# differentes ne decrit aucune d'entre elles.

In [ ]:
verifier("7a - correlation belge", r_be == 0.903, "filtrez d'abord, correlez ensuite")
verifier("7b - correlation irlandaise", r_irl == 0.234,
         "le nom du pays s'ecrit Irlande, avec une majuscule")

### Exercice 8 — Question de synthèse

> **Votre mission :**
> - On vous demande : *« le nombre de produits distincts commandés explique-t-il le montant ? »*
> - Calculer la corrélation de Spearman `ca` / `nart` sur le seul Royaume-Uni → `r_uk` (3 décimales).
> - Puis tracer le nuage de points correspondant.
> - Enfin, écrivez votre réponse en commentaire — en trois phrases maximum.

In [ ]:
uk = cmd.query("pays == 'Royaume-Uni'")   ## le marche de detail
r_uk = round(uk["ca"].corr(uk["nart"], method="spearman"), 3)   ## sur les rangs

uk.plot(kind="scatter", x="nart", y="ca", alpha=0.3, figsize=(7, 4))
plt.xlabel("nart : nombre de produits distincts")
plt.ylabel("ca : montant de la commande (euros)")
plt.show()
print("Spearman au Royaume-Uni :", r_uk)

# Une reponse possible :
# "Oui, le lien est net sur le marche britannique (Spearman 0,57) : les
#  commandes qui portent sur plus de produits distincts sont plus cheres. Mais le
#  coefficient global (0,38) est trompeur, parce qu'il melange des marches
#  de detail et deux grossistes irlandais. Il faut raisonner marche par
#  marche, pas sur le fichier entier."

In [ ]:
verifier("8 - Spearman au Royaume-Uni", r_uk == 0.574,
         "filtrez sur le Royaume-Uni, puis method='spearman'")

---

# Partie 2 — Les questions

Ici, plus de trous : **la cellule sous chaque question est vide**, et c'est à vous
d'écrire le code en entier. C'est exactement ce qu'on vous demandera pour le projet
final, et ce que fait un analyste devant un fichier qu'il découvre.

Certaines questions utilisent une commande que le cours n'a pas montrée. Quand c'est le
cas, l'énoncé vous la donne — savoir se servir d'une commande qu'on vient de lire fait
partie du métier.

> 💡 Pas de vérification automatique dans cette partie. Affichez systématiquement votre
> résultat, et demandez-vous s'il est **plausible** avant de passer à la suite : c'est
> le seul contrôle dont vous disposerez en entreprise.

### Question 9 — Une dépendance certaine, mais forte ?

> **Votre mission :**
> - Le khi-deux dit qu'une dépendance **existe**. Il ne dit rien de sa **force** — exactement comme la p-value du test t.
> - Calculer le **V de Cramér** : `sqrt(khi2 / (n * (k - 1)))`, où `n` est l'effectif total du tab et `k` le plus petit de ses deux nombres de modalités.
> - Il vaut 0 quand il n'y a aucun lien, 1 quand la connaissance de l'un détermine l'autre. Que vaut-il ici ?

In [ ]:
khi2, p, ddl, attendus = stats.chi2_contingency(tab)

n = tab.values.sum()   ## l'effectif total du tableau
k = min(tab.shape)     ## le plus petit nombre de modalites
cramer = np.sqrt(khi2 / (n * (k - 1)))   ## entre 0 et 1

print("p =", p)
print("V de Cramer =", round(cramer, 3))

# p vaut 10 puissance -28 : la dependance est certaine. V vaut 0,21 :
# elle est FAIBLE. Connaitre le pays ameliore un peu la prevision de la
# taille de commande, il ne la determine pas. Les deux nombres racontent
# des choses differentes, et seul le second interesse un dirigeant.

### Question 10 — Quelles cases sont vraiment anormales ?

> **Votre mission :**
> - Le tab des écarts brut (observé − attendu) dépend des effectifs : un écart de 20 est énorme sur une case attendue à 30, négligeable sur une case attendue à 3 000.
> - Calculer les **résidus standardisés** : `(observé - attendu) / sqrt(attendu)`.
> - Repère usuel : au-delà de **2 en valeur absolue**, la case sort de l'ordinaire. Lesquelles ?

In [ ]:
att = pd.DataFrame(attendus, index=tab.index, columns=tab.columns)

residus = (tab - att) / np.sqrt(att)   ## l'ecart, rapporte a l'attendu
residus.round(2)

# Cinq cases sortent du bruit (|residu| > 2) : l'Irlande sur les trois
# tailles (-5,0 ; -3,0 ; +7,8) et le Royaume-Uni sur petites (+3,8) et
# grandes (-5,1). Les deux plus fortes, Irlande/grande et RU/grande,
# disent l'essentiel — grossiste d'un cote, detaillant de l'autre — et
# les cases "petite" le confirment en miroir. Voila la phrase a ecrire
# dans la note — pas "p < 0,001".

### Question 11 — Quand le khi-deux n'est pas applicable

> **Votre mission :**
> - Croiser `jour` et `pays` sur **tous** les pays, puis compter les effectifs **attendus** inférieurs à 5.
> - Recommencer sur les quatre pays les plus présents.
> - Le khi-deux exige des effectifs attendus d'au moins 5. Laquelle des deux p-values a le droit d'être citée ?

In [ ]:
tout = pd.crosstab(cmd["jour"], cmd["pays"])
khi_t, p_t, _, att_t = stats.chi2_contingency(tout)

quatre = pd.crosstab(cmd.query("pays in @top4")["jour"],
                     cmd.query("pays in @top4")["pays"])
khi_q, p_q, _, att_q = stats.chi2_contingency(quatre)

print("23 pays :  p =", round(p_t, 4), "|", (att_t < 5).sum(), "cases attendues sous 5")
print("4 pays  :  p =", round(p_q, 4), "|", (att_q < 5).sum(), "cases attendues sous 5")

# Sur 23 pays, 89 cases sur 138 sont sous le seuil : la p-value de
# 0,0416 ne veut rien dire, elle est calculee hors du domaine de
# validite du test. Sur quatre pays, aucune case ne pose probleme et le
# resultat tient. Un test regroupe les petites modalites AVANT, pas apres.

### Question 12 — Pourquoi Pearson sous-estimait

> **Votre mission :**
> - En partie 1 : Pearson 0,38 contre Spearman 0,67 pour `ca` et `nart`, parce que la relation est **courbe**.
> - Recalculer la corrélation de Pearson sur les **logarithmes** des deux colonnes.
> - Comparer les trois nombres. Qu'est-ce que le logarithme a fait à la relation ?
> - *Nouveau :* `np.log(serie)`.

> 📖 **La transformation logarithmique.** Elle sert à redresser un nuage courbé,
> quand une variable se lit mieux en **proportions** qu'en écarts : passer de 10
> à 20 produits et de 100 à 200 y devient le même pas. Une relation
> multiplicative — ici, doubler le nombre de produits distincts multiplie le
> chiffre d'affaires par environ 1,6 — redevient alors une droite, la seule
> forme que Pearson sache mesurer. Elle exige des valeurs strictement positives.
> [Wikipédia](https://fr.wikipedia.org/wiki/%C3%89chelle_logarithmique)

In [ ]:
print("Pearson brut     :", round(cmd["ca"].corr(cmd["nart"]), 3))
print("Spearman         :", round(cmd["ca"].corr(cmd["nart"], method="spearman"), 3))
# np.log redresse une relation courbe : Pearson retrouve alors sa droite
print("Pearson sur logs :", round(np.log(cmd["ca"]).corr(np.log(cmd["nart"])), 3))

cmd.plot(kind="scatter", x="nart", y="ca", alpha=0.3, loglog=True, figsize=(7, 4))
plt.title("Les memes donnees, sur une echelle logarithmique")
plt.xlabel("nart : nombre de produits distincts")
plt.ylabel("ca : montant de la commande (euros)")
plt.show()

# 0,675 sur les logs, contre 0,671 pour Spearman : les deux methodes
# tombent d'accord. Le logarithme a REDRESSE la courbe ; sur le nuage en
# echelle log, les points s'alignent. Pearson n'avait pas tort, on lui
# demandait de mesurer une droite la ou il n'y en avait pas.

### Question 13 — La corrélation dépend du niveau d'observation

> **Votre mission :**
> - Calculer la corrélation `ca` / `nart` à trois niveaux : par **commande**, puis en **totalisant** le CA et le nombre de produits par **client**, puis par **pays**.
> - Trois nombres très différents à partir des mêmes données.
> - Lequel citeriez-vous dans une note, et pourquoi les deux autres seraient-ils trompeurs ?

In [ ]:
par_client = cmd.groupby("client_id").agg(ca=("ca", "sum"), nart=("nart", "sum"))
par_pays = cmd.groupby("pays").agg(ca=("ca", "sum"), nart=("nart", "sum"))
# Memes donnees, trois niveaux d'observation, trois coefficients

print("par commande :", round(cmd["ca"].corr(cmd["nart"]), 3),
      "sur", len(cmd), "points")
print("par client   :", round(par_client["ca"].corr(par_client["nart"]), 3),
      "sur", len(par_client), "points")
print("par pays     :", round(par_pays["ca"].corr(par_pays["nart"]), 3),
      "sur", len(par_pays), "points")

# 0,38 puis 0,87 puis 0,94. Agreger efface la variabilite individuelle
# et ne laisse que la tendance d'ensemble : la correlation grimpe
# mecaniquement. Une correlation de 0,94 sur 23 points ne dit RIEN du
# comportement d'une commande. On cite le niveau auquel la decision se
# prend, et on precise toujours lequel c'est.

### Question 14 — Conclure proprement à l'absence de lien

> **Votre mission :**
> - Le jour de la semaine influence-t-il la taille des commandes ? Croiser `jour` et `taille`, puis tester (vous devriez trouver p = 0,7866).
> - Ajouter les deux éléments qui manquent pour pouvoir écrire quelque chose : le **V de Cramér**, et le tab des pourcentages par ligne.
> - Rédigez la phrase de conclusion en commentaire. Attention à ne pas écrire « il n'y a pas de lien ».

In [ ]:
tab_jour = pd.crosstab(cmd["jour"], cmd["taille"])
khi_j, p_j, _, att_j = stats.chi2_contingency(tab_jour)

n = tab_jour.values.sum()
v = np.sqrt(khi_j / (n * (min(tab_jour.shape) - 1)))   ## la FORCE du lien
print("p =", round(p_j, 4), "| V de Cramer =", round(v, 3))

(pd.crosstab(cmd["jour"], cmd["taille"], normalize="index") * 100).round(1)

# Phrase possible :
# "La repartition des tailles de commande est stable d'un jour a
#  l'autre : entre 24 et 30 % de petites commandes selon le jour, et
#  aucun ecart que le hasard n'expliquerait (p = 0,79 ; V = 0,04). Ces
#  donnees ne mettent en evidence aucun effet du jour de la semaine."
# Ce n'est pas "il n'y a pas d'effet" : c'est "on n'en detecte pas".

### Question 15 — Question de synthèse

> **Votre mission :**
> - On vous demande : *« le pays détermine-t-il la façon d'acheter ? »*
> - Répondez avec trois éléments : l'existence du lien (p), sa force (V de Cramér), et l'endroit où il se situe (les résidus).
> - Puis, en commentaire, la conclusion en trois phrases — dont une qui dit ce que ces données **ne** permettent **pas** d'affirmer.

In [ ]:
khi2, p, ddl, attendus = stats.chi2_contingency(tab)
att = pd.DataFrame(attendus, index=tab.index, columns=tab.columns)

print("existence :  p =", p)
print("force     :  V =", round(np.sqrt(khi2 / (tab.values.sum() * (min(tab.shape) - 1))), 3))
print("localisation :")
print(((tab - att) / np.sqrt(att)).round(1))

# Conclusion possible :
# "Oui, la taille des commandes depend du marche (p < 0,001), mais le
#  lien est faible (V = 0,21) : le pays n'explique qu'une petite part du
#  comportement d'achat. Tout se joue sur deux marches — l'Irlande achete
#  gros (+7,8 sur les grandes commandes), le Royaume-Uni achete petit
#  (+3,8 sur les petites, -5,1 sur les grandes) — tandis que l'Allemagne
#  et la France sont conformes a ce qu'on attendrait au hasard. Ces
#  donnees ne disent pas POURQUOI :
#  l'Irlande n'a que deux clients, tous deux grossistes, et c'est le
#  type de client qui explique l'achat, pas la geographie."